# Silver-to-Gold data transformations

This notebook is a pipeline orchestration driver that controls the execution of Silver-to-Gold data transformations in a Databricks banking project.

It does not perform data transformation itself. Instead, it:

- receives execution parameters
- manages metadata and audit logging
- triggers Gold transformation notebooks dynamically
- tracks execution status (SUCCESS / FAILED)

The main goal is to ensure controlled, traceable, and monitored Gold-layer processing.

In [0]:
import json
from datetime import datetime

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("table_metadata", "")

run_id = dbutils.widgets.get("run_id")
table_metadata = dbutils.widgets.get("table_metadata")

if not run_id or not table_metadata:
    raise ValueError("run_id and table_metadata are required")
table_metadata = json.loads(table_metadata.replace("'", '"'))

table_id = table_metadata["table_id"]
table_name = table_metadata["table_name"]

start_time = datetime.utcnow()

print("Run ID:", run_id)
print("Table ID:", table_id)
print("Table Name:", table_name)

In [0]:
%sql
create schema if not exists banking.gold;

In [0]:
# Audit and Execution Control
# This block manages audit logging, executes the
# Gold transformation notebook, and updates the final execution status.

# Check whether an audit entry already exists
entry_exists = spark.sql(f"""
    SELECT 1
    FROM banking.metadata.pipeline_runs
    WHERE run_id = {run_id}
      AND table_id = {table_id}
""").count() > 0

if entry_exists:

    spark.sql(f"""
        UPDATE banking.metadata.pipeline_runs
        SET
            layer = 'Gold',
            start_time = TIMESTAMP('{start_time}'),
            end_time = NULL,
            status = 'INPROGRESS',
            number_of_records = NULL,
            error_message = NULL
        WHERE run_id = {run_id}
          AND table_id = {table_id}
    """)

else:

    spark.sql(f"""
        INSERT INTO banking.metadata.pipeline_runs
        VALUES (
            {run_id},
            {table_id},
            'Gold',
            TIMESTAMP('{start_time}'),
            NULL,
            'INPROGRESS',
            NULL,
            NULL
        )
    """)

print("Audit entry created / updated")

In [0]:
# Build the notebook path dynamically
notebook_path = f"gold_transformations/{table_name}"

print("Notebook to execute:", notebook_path)

In [0]:
# Initialize execution status variables
status = "SUCCESS"
error_message = None
records = None

try:

    result = dbutils.notebook.run(
        notebook_path,
        timeout_seconds=0
    )

    if result:
        records = int(result)

    print("Notebook completed successfully")
    print("Records:", records)

except Exception as e:

    status = "FAILED"
    error_message = str(e)

    print("Notebook failed")
    print(error_message)

In [0]:
# Capture execution end time
end_time = datetime.utcnow()

# Update audit table with final execution status
spark.sql(f"""
    UPDATE banking.metadata.pipeline_runs
    SET
        end_time = TIMESTAMP('{end_time}'),
        status = '{status}',
        number_of_records = {records if records else 'NULL'},
        error_message = {f"'{error_message}'" if error_message else 'NULL'}
    WHERE run_id = {run_id}
      AND table_id = {table_id}
""")

print("Audit table updated")

In [0]:
# Raise an exception if the notebook execution failed
if status == "FAILED":
    raise Exception(error_message)